# 02 — Reasoning-Text Entropy: GPU-Accelerated Confirmation

**Follows `pilot.ipynb`, but is independent of it** -- this notebook does not
load Qwen2.5-VL or touch the FERMAT dataset at all. It only needs the grading
prompt's raw text output, already collected and saved by `pilot.ipynb`'s main
run (K=5) and the grading K-resample run (K=15). No `HF_TOKEN` needed.

**What this confirms.** The grading pipeline currently reduces each sample to
a single parsed `**Error:**` digit before computing entropy. Analysis on the
K=5 baseline this session found: digit-only reasoning entropy gives
AUROC=0.618 with an *exact* median tie between correct/incorrect groups.
Clustering the `**Reasoning:**` explanation text instead (via bidirectional
NLI entailment) gives AUROC=0.702 on the same 100 items -- a real
improvement, verified not to be an artifact of NLI's known lexical-overlap
bias: spot-checking cases where two samples merged despite disagreeing on the
digit showed the model's own reasoning was self-contradicting its digit in
~12% of samples (60/494) -- the digit is noisier than the reasoning text, not
the other way around.

**Why GPU / why this notebook.** NLI clustering is O(K^2) pairwise
comparisons per item. On local CPU, a single K=15 item's 210 pairs took ~35s
-- a full 100-item K=15 pass would take 1-2 hours locally. On a Colab GPU
this should be a small fraction of that. This notebook re-confirms the K=5
result (should reproduce AUROC=0.702) and runs the K=15 confirmation that
was impractical locally, on the full 100 items for both.

In [ ]:
# Install cell. Deliberately minimal -- no transformers/accelerate/
# bitsandbytes/qwen-vl-utils, since this notebook never loads the VLM.
!pip install -q sentence-transformers pandas numpy

In [ ]:
# Auth & code access cell. No HF_TOKEN needed -- this notebook never
# touches the gated FERMAT dataset, only already-collected result CSVs.
import json
import os
from getpass import getpass

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_RESULTS_DIR = f"{PROJECT_DIR}/results"

# --- GitHub token: reused from the same Drive-cached store pilot.ipynb uses,
# only needed for the save cell's push at the end. ---
TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

# --- Clone the repo (anonymous read -- public repo, token used only to push later) ---
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"

!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.parsing
import pilot.entropy
import pilot.semantic
import pilot.plotting

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

In [ ]:
# Load the NLI model on GPU, and the two result CSVs to analyze.
import torch

print("CUDA available:", torch.cuda.is_available())

from sentence_transformers import CrossEncoder

nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small", device="cuda" if torch.cuda.is_available() else "cpu")

import pandas as pd

BASELINE_CSV = f"{DRIVE_RESULTS_DIR}/results_qwen25-vl-3b-instruct_20260731T210745Z.csv"
KRESAMPLE_CSV = f"{DRIVE_RESULTS_DIR}/grading_kresample_k15_qwen25-vl-3b-instruct_20260801T215042Z.csv"

baseline_df = pd.read_csv(BASELINE_CSV)
kresample_df = pd.read_csv(KRESAMPLE_CSV)
print(f"baseline (K=5): {len(baseline_df)} rows")
print(f"kresample (K=15): {len(kresample_df)} rows")

In [ ]:
# Compute reasoning-text entropy for every item at both K=5 and K=15,
# timing each pass so the GPU speedup vs. the local CPU estimate is visible.
import ast
import time

from tqdm.auto import tqdm


def reasoning_text_entropy_for_row(raw_samples_json, model):
    raws = ast.literal_eval(raw_samples_json)
    reasonings = [pilot.parsing.parse_grading_reasoning(r) for r in raws]
    labels = pilot.semantic.nli_cluster_labels(reasonings, _model=model)
    return pilot.entropy.entropy_from_labels(labels), reasonings, labels


def run_pass(df, raw_col, k_label):
    entropies = []
    t0 = time.time()
    for i in tqdm(range(len(df)), desc=f"reasoning-text entropy K={k_label}"):
        ent, _, _ = reasoning_text_entropy_for_row(df[raw_col].iloc[i], nli_model)
        entropies.append(ent)
    elapsed = time.time() - t0
    print(f"K={k_label}: {len(df)} items in {elapsed:.0f}s ({elapsed / len(df):.1f}s/item)")
    return entropies


baseline_df["reasoning_text_entropy_k5"] = run_pass(baseline_df, "all_grading_samples_raw", 5)
kresample_df["reasoning_text_entropy_k15"] = run_pass(kresample_df, "all_grading_samples_raw_k15", 15)

In [ ]:
# AUROC comparison: digit-only (already computed by earlier runs) vs
# reasoning-text (computed above), at both K levels.
auroc_digit_k5 = pilot.plotting.compute_auroc(baseline_df, "reasoning_entropy", "grading_correct")
auroc_text_k5 = pilot.plotting.compute_auroc(baseline_df, "reasoning_text_entropy_k5", "grading_correct")
auroc_digit_k15 = pilot.plotting.compute_auroc(kresample_df, "reasoning_entropy_k15", "grading_correct_k15")
auroc_text_k15 = pilot.plotting.compute_auroc(kresample_df, "reasoning_text_entropy_k15", "grading_correct_k15")

print("=== AUROC comparison ===")
print(f"digit-only          K=5:  {auroc_digit_k5:.4f}")
print(f"reasoning-text       K=5:  {auroc_text_k5:.4f}  (local CPU run found 0.7019 -- should reproduce)")
print(f"digit-only          K=15: {auroc_digit_k15:.4f}")
print(f"reasoning-text       K=15: {auroc_text_k15:.4f}  (new -- was impractical locally)")

### Validation note

The improvement above is only trustworthy if it is not the same lexical-overlap
over-merging bug found earlier in the perception arm (see `pilot.ipynb` and the
project report). The check below verifies this the same way it was verified
locally on K=5: find pairs of samples that were merged into the same
reasoning-text cluster despite disagreeing on the parsed digit, and inspect
whether the merge is a genuine digit/reasoning self-contradiction (both texts
argue the same direction, one sample's own digit is just wrong) rather than the
model's own reasoning texts genuinely opposing each other.

In [ ]:
# Cross-digit merge validation, run on BOTH K levels this time (only K=5
# was checked locally). A high rate here with genuinely opposed reasoning
# texts would be a red flag; a high rate where texts agree in substance
# despite differing digits confirms the earlier finding generalizes.
def cross_digit_merge_check(df, raw_col, digit_col_exists_in_raw=True):
    cross_merges = 0
    items_with_split = 0
    examples = []
    for i in tqdm(range(len(df)), desc="cross-digit check"):
        raws = ast.literal_eval(df[raw_col].iloc[i])
        reasonings = [pilot.parsing.parse_grading_reasoning(r) for r in raws]
        digits = [pilot.parsing.parse_grading(r) for r in raws]
        if len(set(d for d in digits if d is not None)) < 2:
            continue
        items_with_split += 1
        labels = pilot.semantic.nli_cluster_labels(reasonings, _model=nli_model)
        for a in range(len(digits)):
            for b in range(a + 1, len(digits)):
                if (
                    labels[a] == labels[b]
                    and digits[a] is not None
                    and digits[b] is not None
                    and digits[a] != digits[b]
                ):
                    cross_merges += 1
                    if len(examples) < 5:
                        examples.append((i, digits[a], digits[b], reasonings[a], reasonings[b]))
    return items_with_split, cross_merges, examples


print("=== K=5 ===")
items_k5, merges_k5, examples_k5 = cross_digit_merge_check(baseline_df, "all_grading_samples_raw")
print(f"items with digit split: {items_k5}/100, cross-digit merges: {merges_k5}")

print()
print("=== K=15 ===")
items_k15, merges_k15, examples_k15 = cross_digit_merge_check(kresample_df, "all_grading_samples_raw_k15")
print(f"items with digit split: {items_k15}/100, cross-digit merges: {merges_k15}")

print()
print("=== sample K=15 cross-digit merges for manual inspection ===")
for i, da, db, ra, rb in examples_k15[:5]:
    print(f"--- item {i}: digit={da} vs digit={db} ---")
    print(f"  [{da}]: {ra[:200]!r}")
    print(f"  [{db}]: {rb[:200]!r}")
    print()

In [ ]:
# Save results + validation summary, commit, push. Same pattern as
# pilot.ipynb's save cells: Drive backup written before any git operation,
# token redacted from any printed git output, fetch+rebase before push.
import subprocess
from datetime import datetime, timezone

summary_df = pd.DataFrame(
    [
        {
            "k": 5, "auroc_digit_only": auroc_digit_k5, "auroc_reasoning_text": auroc_text_k5,
            "items_with_digit_split": items_k5, "cross_digit_merges": merges_k5,
        },
        {
            "k": 15, "auroc_digit_only": auroc_digit_k15, "auroc_reasoning_text": auroc_text_k15,
            "items_with_digit_split": items_k15, "cross_digit_merges": merges_k15,
        },
    ]
)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
summary_name = f"reasoning_text_entropy_summary_{timestamp}.csv"
per_item_k5_name = f"reasoning_text_entropy_k5_{timestamp}.csv"
per_item_k15_name = f"reasoning_text_entropy_k15_{timestamp}.csv"

os.makedirs("repo/results", exist_ok=True)
summary_df.to_csv(f"repo/results/{summary_name}", index=False)
baseline_df.to_csv(f"repo/results/{per_item_k5_name}", index=False)
kresample_df.to_csv(f"repo/results/{per_item_k15_name}", index=False)

for name, df_to_save in (
    (summary_name, summary_df), (per_item_k5_name, baseline_df), (per_item_k15_name, kresample_df)
):
    df_to_save.to_csv(f"{DRIVE_RESULTS_DIR}/{name}", index=False)
print(f"Wrote and backed up: {summary_name}, {per_item_k5_name}, {per_item_k15_name}")

_REDACT = [GH_TOKEN]


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{summary_name}", f"results/{per_item_k5_name}", f"results/{per_item_k15_name}")
commit = git("commit", "-m", f"Add reasoning-text entropy confirmation (K=5 and K=15): {summary_name}")
if commit.returncode != 0:
    raise RuntimeError("git commit failed -- see output above")
print("Committed.")

push_url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")
if git("fetch", push_url, "main").returncode == 0:
    if git("rebase", "FETCH_HEAD").returncode != 0:
        git("rebase", "--abort")
        print("Rebase onto remote failed; attempting push anyway.")
if git("push", push_url, "HEAD:main").returncode == 0:
    print("Pushed results to the repo.")
else:
    print(
        "Push failed (see output above -- if 403, GH_TOKEN lacks write access; "
        "regenerate with 'repo' scope). Results are safe on Drive and in repo/results/ either way."
    )